In [111]:
import glob
import os
import xarray as xa
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import numpy as np
from scipy.interpolate import RectBivariateSpline
from scipy.stats import pearsonr
from scipy.ndimage import convolve

import cartopy.crs as ccrs
from cartopy.feature import LAND, COASTLINE
import cmocean.cm as ocm

from subprocess import call
import shutil


In [3]:
band_names = [
    'SI_12km_NH_18H_ASC',
    'SI_12km_NH_18V_ASC',
    'SI_12km_NH_23H_ASC',
    'SI_12km_NH_23V_ASC',
    'SI_12km_NH_36H_ASC',
    'SI_12km_NH_36V_ASC',
    'SI_12km_NH_89H_ASC',
    'SI_12km_NH_89V_ASC',
]
with xa.open_dataset('/data1/antonk/tmp/AMSR_U2_L3_SeaIce12km_B04_20230101.he5', group='HDFEOS/GRIDS/NpPolarGrid12km/Data Fields') as ds:
    bands = np.dstack([ds[band_name].to_numpy() for band_name in band_names])


In [ ]:
gpi = np.isfinite(bands[:,:,0])
bands_gpi = bands[gpi, :]
corr_coef = np.corrcoef(bands_gpi.T)
plt.imshow(corr_coef, cmap='jet')
plt.colorbar()
plt.show()

bands_avg = bands_gpi.mean(axis=0)
bands_std = bands_gpi.std(axis=0)
bands_sca = np.clip(255 * (0.5 + (bands - bands_avg[None][None]) / bands_std[None][None] / 5), 0, 255).astype('uint8')
plt.imshow(bands_sca[:,:,[0,6,7]])
plt.show()

In [5]:
with xa.open_dataset('/data1/antonk/tmp/AMSR_U2_L3_SeaIce12km_B04_20230101.he5', group='HDFEOS/GRIDS/NpPolarGrid12km') as ds:
    lon = ds.lon.to_numpy()
    lat = ds.lat.to_numpy()

In [6]:
titles = {
    'siconc': 'Concentration',
    'sithick': 'Thickness',
    'siconc_my': 'MYI concentration',
    'siage': 'Ice Age',
    'si_ridge_ratio': 'Ridged ice (roughness)',
    'sisnthick': 'Snow thickness',
}
ifile = '/data1/antonk/nextsim_reanalysis/2023/01/20230101_dm-nersc-MODEL-nextsimf-ARC-RAN-fv00.0.nc'
with xa.open_dataset(ifile) as ds:
    data = {name: ds[name][0].to_numpy() for name in titles}
    x = ds.x.to_numpy()
    y = ds.y.to_numpy()
data['mask'] = np.isnan(data['sithick'])
data['mask'][0] = True
titles['mask'] = 'Landmask'


In [7]:
proj = ccrs.NorthPolarStereo(true_scale_latitude=90, central_longitude=-45)
amsr_x, amsr_y, _ = proj.transform_points(ccrs.PlateCarree(), lon, lat).T
amsr_x, amsr_y = amsr_x.T, amsr_y.T

In [ ]:
amsr_data = {}
for name in titles:
    data[name] = np.where(np.isnan(data[name]), 0, data[name])
    s = RectBivariateSpline(y, x, data[name])
    amsr_data[name] = s(amsr_y, amsr_x, grid=False)
    print(name)

In [9]:
for name in titles:
    if name != 'mask':
        amsr_data[name][amsr_data['mask'] > 0.1] = 0
        amsr_data[name][amsr_data[name] < 0.01] = 0

In [10]:
cube = np.dstack([bands, np.dstack([amsr_data[name] for name in titles])])
gpi = np.isfinite(cube[:,:,0]) * (cube[:, :, 14] < 0.5) #* (cube[:, :, 8] > 0.001)
cube_gpi = cube[gpi, :]

In [ ]:
corr_coef_cube = np.corrcoef(cube_gpi.T)
plt.imshow(corr_coef_cube, cmap='jet')
plt.colorbar()
plt.show()

In [ ]:
odir = '/data1/antonk/nextsim_reanalysis/2023/pngs'

A0 = np.vstack([cube_gpi[:, 8], cube_gpi[:, 9], np.ones_like(cube_gpi[:, 8])]).T
A1 = np.vstack([cube_gpi[:, 8], cube_gpi[:, 9], cube_gpi[:, 10], np.ones_like(cube_gpi[:, 8])]).T
A2 = np.vstack([cube_gpi[:, 8], cube_gpi[:, 9], cube_gpi[:, 10], cube_gpi[:, 11], np.ones_like(cube_gpi[:, 8])]).T
A3 = np.vstack([cube_gpi[:, 8], cube_gpi[:, 9], cube_gpi[:, 10], cube_gpi[:, 11], cube_gpi[:, 12], np.ones_like(cube_gpi[:, 8])]).T
A4 = np.vstack([cube_gpi[:, 8], cube_gpi[:, 9], cube_gpi[:, 10], cube_gpi[:, 11], cube_gpi[:, 12], cube_gpi[:, 13], np.ones_like(cube_gpi[:, 8])]).T

bands_pred = {i:np.array(bands) for i in range(5)}

BB = {}

for model_no, A in enumerate([A0, A1, A2, A3, A4]):
    for i, j in enumerate([0, 6, 7]):
        B = np.linalg.lstsq(A, cube_gpi[:, j])[0]
        BB[f'{model_no}_{j}'] = B
        Y = np.dot(A, B)
        bands_pred[model_no][gpi, j] = Y.flatten()
        r = pearsonr(Y, cube_gpi[:, j])[0]
        lims = np.percentile(cube_gpi[:, j], [1, 99])
        fig, axs = plt.subplots(1, 1, figsize=(5,5))
        axs.hist2d(cube_gpi[:, j], Y, 100, [lims, lims], cmin=1, norm=LogNorm())
        axs.plot(lims, lims, 'k-')
        axs.set_title(f'{r:0.3}')
        ofile = f'{odir}/scater_mod{model_no}_band{j}.png'
        print(ofile)
        plt.savefig(ofile, dpi=100, pad_inches=0.1, bbox_inches='tight')
        plt.close()

In [ ]:
model_titles = ['SIC,SIT', 'SIC,SIT,MYI', 'SIC,SIT,MYI,SIA', 'SIC,SIT,MYI,SIA,SIR', 'SIC,SIT,MYI,SIA,SIR,SNT']
for model_no in range(5):
    bands_pred_sca = np.clip(255 * (0.5 + (bands_pred[model_no] - bands_avg[None][None]) / bands_std[None][None] / 5), 0, 255).astype('uint8')
    fig, axs = plt.subplots(1, 3, figsize=(15, 6))
    axs[0].imshow(bands_sca[:,:,[0,6,7]], extent=[amsr_x.min(), amsr_x.max(), amsr_y.min(), amsr_y.max()])
    axs[1].imshow(bands_pred_sca[:,:,[0,6,7]], extent=[amsr_x.min(), amsr_x.max(), amsr_y.min(), amsr_y.max()])
    axs[2].imshow(bands_pred[model_no][:,:,0] - cube[:,:,0], extent=[amsr_x.min(), amsr_x.max(), amsr_y.min(), amsr_y.max()], clim=[-70, 70], cmap='coolwarm')
    axs[0].set_title('AMSR2')
    axs[1].set_title(model_titles[model_no])
    for ax in axs:
        ax.set_xlim([-2.5e6, 1.5e6])
        ax.set_ylim([-2e6, 2e6])
    ofile = f'{odir}/amsr2_maps_mo{model_no}.png'
    plt.savefig(ofile, dpi=150, bbox_inches='tight', pad_inches=0.1)
    plt.show()
    print(ofile)


In [123]:
ifiles = sorted(glob.glob('/data1/antonk/nextsim_reanalysis/2023/0[2-4]/*.nc'))

In [ ]:
odir = '/data1/antonk/nextsim_reanalysis/2023/pngs'

titles = {
    'siconc': 'Concentration',
    'sithick': 'Thickness',
    'siconc_my': 'MYI concentration',
    'siage': 'Ice Age',
    'si_ridge_ratio': 'Ridged ice (roughness)',
    'sisnthick': 'Snow thickness',
}

for ifile in ifiles:
    datestr = os.path.basename(ifile).split('_')[0]
    ofile = f'{odir}/rgb_{datestr}.png'
    if os.path.exists(ofile):
        continue

    try:
        with xa.open_dataset(ifile) as ds:
            data = {name: ds[name][0].to_numpy() for name in titles}
            x = ds.x.to_numpy()
            y = ds.y.to_numpy()
    except:
        continue

    A3_full = np.vstack([
        data['siconc'].flatten(),
        data['sithick'].flatten(),
        data['siconc_my'].flatten(),
        data['siage'].flatten(),
        data['si_ridge_ratio'].flatten(),
        np.ones(data['siconc'].size)
    ]).T

    Y_pred = []
    for i, j in enumerate([0, 6, 7]):
        y_pred = np.dot(A3_full, BB[f'3_{j}']).reshape(data['siconc'].shape)
        y_pred_sca = np.clip(255 * (0.5 + (y_pred - bands_avg[j]) / bands_std[j] / 5), 0, 255).astype('uint8')
        Y_pred.append(y_pred_sca)
    Y_pred = np.dstack(Y_pred)

    cimr_weights = np.array([[0.5,0.5],[0,0]])
    amsr_weights = np.ones((8,5)) / 8 / 5

    Y_pred_cimr = np.dstack([convolve(Y_pred[:,:,i], cimr_weights) for i in range(3)])
    Y_pred_amsr = np.dstack([convolve(Y_pred[:,:,i], amsr_weights) for i in range(3)])

    fig, axs = plt.subplots(1, 2, figsize=(10,5), sharey=True, subplot_kw={'projection': proj},)
    axs[0].imshow(
        Y_pred_amsr,
        extent=[x[0], x[-1], y[0], y[-1]],
        origin='lower',
        zorder = 0,
    )
    axs[1].imshow(
        Y_pred_cimr,
        extent=[x[0], x[-1], y[0], y[-1]],
        origin='lower',
        zorder = 0,
    )
    for ax in axs:
        ax.set_xlim([0, 1.5e6])
        ax.set_ylim([-1.5e6, 0])
        ax.add_feature(LAND, zorder=1)
        ax.add_feature(COASTLINE, zorder=2)
    plt.tight_layout()
    plt.savefig(ofile, bbox_inches='tight', pad_inches=0.1, dpi=150)
    plt.close()


In [126]:
name = 'rgb'
png_files = sorted(glob.glob(f'{odir}/{name}*png'))
frame_files = sorted(glob.glob(f'{odir}/frames/*png'))
[os.remove(f) for f in frame_files]
for i, png_file in enumerate(png_files):
    frame_name = f'{odir}/frames/frame_{i:05}.png'
    shutil.copy(png_file, frame_name)


In [ ]:
cmd = f"ffmpeg -y -r 3 -i {odir}/frames/frame_%05d.png -s 1472x734 -r 25 -c:v libx264 -crf 20  -pix_fmt yuv420p ~/{name}.mov"
print(cmd)
call(cmd, shell=True)